# ForestWatch â€” FoSp on SMOKE5K
Official FoSp architecture, audited dataset, train-only validation and resumable training. No claim of 80% until measured.
Enable GPU and Internet. Run cells in order. For a new session, attach saved outputs and set RESUME_FROM below.

In [ ]:
from pathlib import Path
Path("/kaggle/working/smoke5k-code").mkdir(exist_ok=True)

In [ ]:
%%writefile /kaggle/working/smoke5k-code/prepare_smoke5k.py
"""Audit the authors' SMOKE5K archive and create a train-only validation split.

Run after extracting the official archive. Original images and masks stay intact.
The supplied archive currently contains 5,058 training pairs and 400 test pairs;
the manifest records observed counts rather than assuming a paper's rounded size.
"""
import argparse
import collections
import hashlib
import json
import re
from pathlib import Path

import numpy as np
from PIL import Image


def group_key(name, digest):
    # These names include a frame timestamp and elapsed seconds. Hold an entire
    # inferred recording together; opaque annotation IDs remain separate groups.
    match = re.fullmatch(r"(\d{10})_\+(\d+)", name)
    if match:
        return "recording:" + str(int(match[1]) - int(match[2]))
    return "pixels:" + digest


def prepare(root, output, seed=42):
    output.mkdir(parents=True, exist_ok=True)
    entries, report = {}, {}
    for split, mask_dir in (("train", "gt"), ("test", "gt_")):
        def index(directory):
            result = {}
            for path in directory.iterdir():
                if path.name.startswith(".") or path.suffix.lower() not in {".png", ".jpg", ".jpeg"}:
                    continue
                if path.stem in result:
                    raise ValueError(f"Duplicate stem: {path}")
                result[path.stem] = path
            return result

        images = index(root / split / "img")
        masks = index(root / split / mask_dir)
        if set(images) != set(masks):
            raise ValueError(f"Unmatched {split} stems: {set(images) ^ set(masks)}")
        rows, values, digests = [], collections.Counter(), collections.defaultdict(list)
        for number, name in enumerate(sorted(images), 1):
            with Image.open(images[name]) as source:
                rgb = source.convert("RGB")
                digest = hashlib.sha256(rgb.tobytes() + str(rgb.size).encode()).hexdigest()
                size = rgb.size
            with Image.open(masks[name]) as source:
                if source.size != size:
                    raise ValueError(f"Image/mask dimensions differ: {name}")
                unique = set(map(int, np.unique(np.array(source.convert("L")))))
                if not unique.issubset({0, 255}):
                    raise ValueError(f"Unexpected mask encoding for {name}: {unique}")
                values.update(unique)
            digests[digest].append(name)
            rows.append({"id": name, "image": str(images[name]), "mask": str(masks[name]),
                         "pixel_sha256": digest, "group": group_key(name, digest)})
            if number % 1000 == 0:
                print("AUDIT_PROGRESS", split, number, flush=True)
        entries[split] = rows
        report[split] = {"pairs": len(rows), "mask_value_image_counts": dict(values),
                         "duplicate_copies": sum(len(v) - 1 for v in digests.values())}

    test_hashes = {row["pixel_sha256"] for row in entries["test"]}
    excluded = [row for row in entries["train"] if row["pixel_sha256"] in test_hashes]
    eligible = [row for row in entries["train"] if row["pixel_sha256"] not in test_hashes]
    train, val = [], []
    for row in eligible:
        number = int(hashlib.sha256(f"{seed}:{row['group']}".encode()).hexdigest()[:16], 16)
        (val if number % 10 == 0 else train).append(row)
    assert train and val and entries["test"]
    assert not ({r["group"] for r in train} & {r["group"] for r in val})
    assert not ({r["pixel_sha256"] for r in train + val} & test_hashes)
    report.update({"excluded_train_test_exact_duplicates": [r["id"] for r in excluded],
                   "seed": seed, "train": report["train"] | {"used_for_train": len(train)},
                   "validation_pairs": len(val), "official_test_pairs": len(entries["test"]),
                   "validation_policy": "10% hash partition of training groups; seed 42",
                   "grouping_limit": "Timestamp/offset recordings inferred from names; opaque IDs and camera near-duplicates are not fully audited.",
                   "mask_mapping": "0 -> background 0; 255 -> smoke 1. No ignore pixels.",
                   "source": "https://github.com/SiyuanYan1/Transmission-BVM",
                   "source_archive_id": "11TM8hsh9R6ZTvLAUzfD6eD051MbOufCi",
                   "paper_reproduction": False,
                   "protocol_note": "Internal validation holdout and exact-overlap removal differ from training on the entire official train folder."})
    manifest = {"root": str(root), "report": report, "train": train,
                "val": val, "test": entries["test"]}
    (output / "manifest.json").write_text(json.dumps(manifest, indent=2))
    (output / "data_audit.json").write_text(json.dumps(report, indent=2))
    print("SMOKE5K_PREPARED", json.dumps(report), flush=True)
    return manifest


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--root", type=Path, default=Path("/kaggle/working/SMOKE5K-official/SMOKE5K"))
    parser.add_argument("--output", type=Path, default=Path("/kaggle/working/fosp_smoke5k"))
    args = parser.parse_args()
    prepare(args.root, args.output)


In [ ]:
%%writefile /kaggle/working/smoke5k-code/train_smoke5k.py
"""Train official FoSp on audited SMOKE5K; retain resumable checkpoints.

Compatibility adaptation: reference MAT ops, microbatch gradient accumulation,
train-only validation holdout and an explicit time budget. Not an exact paper
reproduction. Test is evaluated only by an explicit --evaluate-test invocation.
"""
import argparse
import hashlib
import json
import logging
import os
import random
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as functional
from PIL import Image, ImageDraw, ImageEnhance
from mmcv import Config
from mmseg.models import build_segmentor

MEAN = np.array([123.675, 116.28, 103.53], dtype=np.float32)
STD = np.array([58.395, 57.12, 57.375], dtype=np.float32)


def load_pair(row, augment=False):
    with Image.open(row["image"]) as source:
        image = source.convert("RGB").resize((512, 512), Image.Resampling.BILINEAR)
    with Image.open(row["mask"]) as source:
        # SMOKE5K uses white (255) for smoke, NOT for ignore_index.
        mask = source.convert("L").resize((512, 512), Image.Resampling.NEAREST)
    if augment:
        # The same geometric transform is applied to both image and mask.
        if random.random() < 0.5:
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            mask = mask.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        # Mild zoom crops retain most context; no synthetic smoke is generated.
        side = random.randint(410, 512)
        left, top = random.randint(0, 512 - side), random.randint(0, 512 - side)
        box = (left, top, left + side, top + side)
        image = image.crop(box).resize((512, 512), Image.Resampling.BILINEAR)
        mask = mask.crop(box).resize((512, 512), Image.Resampling.NEAREST)
        if random.random() < 0.5:
            image = ImageEnhance.Brightness(image).enhance(random.uniform(0.85, 1.15))
            image = ImageEnhance.Contrast(image).enhance(random.uniform(0.85, 1.15))
    pixels = np.array(image, dtype=np.float32)
    target = (np.array(mask) > 127).astype(np.int64)
    x = torch.from_numpy(((pixels - MEAN) / STD).transpose(2, 0, 1).copy())
    y = torch.from_numpy(target).unsqueeze(0)
    return x, y


def metrics_from_counts(tp, fp, fn):
    return {"smoke_iou": tp / max(tp + fp + fn, 1),
            "f1": 2 * tp / max(2 * tp + fp + fn, 1),
            "precision": tp / max(tp + fp, 1), "recall": tp / max(tp + fn, 1)}


@torch.no_grad()
def evaluate(model, rows, threshold, output, label, preview=False):
    """Report both image-mean and global-pixel metrics at a fixed threshold.

    Predictions are resized back to each native ground-truth resolution before
    scoring. F1 has beta=1; it is not assumed identical to every paper's F_beta.
    """
    model.eval()
    numpy_state = np.random.get_state()
    np.random.seed(123)
    total = np.zeros(3, dtype=np.int64)
    per_image = []
    canvas = Image.new("RGB", (768, 4 * 280), "white") if preview else None
    try:
        for index, row in enumerate(rows):
            x, _ = load_pair(row)
            logits = model.encode_decode(x.unsqueeze(0).cuda(), [{}])
            with Image.open(row["mask"]) as source:
                target = np.array(source.convert("L")) > 127
            probability = torch.sigmoid(functional.interpolate(
                logits, size=target.shape, mode="bilinear", align_corners=False))[0, 0].cpu().numpy()
            pred = probability > threshold
            tp = int(np.count_nonzero(pred & target))
            fp = int(np.count_nonzero(pred & ~target))
            fn = int(np.count_nonzero(~pred & target))
            total += (tp, fp, fn)
            score = metrics_from_counts(tp, fp, fn)
            score["mae"] = float(np.abs(probability - target.astype(np.float32)).mean())
            per_image.append({"id": row["id"], **score})
            if canvas is not None and index < 4:
                with Image.open(row["image"]) as source:
                    rgb = source.convert("RGB")
                pictures = [rgb]
                for mask, color in ((target, [60, 220, 90]), (pred, [255, 165, 45])):
                    pixels = np.array(rgb).copy()
                    pixels[mask] = (0.5 * pixels[mask] + 0.5 * np.array(color)).astype(np.uint8)
                    pictures.append(Image.fromarray(pixels))
                draw = ImageDraw.Draw(canvas)
                for col, (title, picture) in enumerate(zip(("Source", "True smoke (green)", "FoSp (orange)"), pictures)):
                    draw.text((col * 256 + 6, index * 280 + 6), title, fill="black")
                    canvas.paste(picture.resize((256, 256)), (col * 256, index * 280 + 24))
            if (index + 1) % 100 == 0:
                print("EVAL_PROGRESS", label, index + 1, len(rows), flush=True)
    finally:
        np.random.set_state(numpy_state)
    scores = {"n_images": len(rows), "threshold": threshold,
              "image_mean": {key: float(np.mean([r[key] for r in per_image]))
                             for key in ("smoke_iou", "f1", "precision", "recall", "mae")},
              "global_pixels": metrics_from_counts(*map(int, total))}
    (output / f"{label}_per_image.json").write_text(json.dumps(per_image, indent=2))
    if canvas is not None:
        canvas.save(output / f"{label}_preview.jpg", quality=90)
    return scores


def atomic_save(value, path):
    temporary = path.with_suffix(".tmp")
    torch.save(value, temporary)
    os.replace(temporary, path)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--output", type=Path, default=Path("/kaggle/working/fosp_smoke5k"))
    parser.add_argument("--weights", type=Path, default=Path("/kaggle/working/fosp-pretrained"))
    parser.add_argument("--max-updates", type=int, default=40000)
    parser.add_argument("--micro-batch", type=int, default=2)
    parser.add_argument("--accumulate", type=int, default=3)
    parser.add_argument("--time-budget", type=int, default=28800)
    parser.add_argument("--eval-every", type=int, default=500)
    parser.add_argument("--resume", action="store_true")
    parser.add_argument("--evaluate-test", action="store_true")
    args = parser.parse_args()
    if min(args.max_updates, args.micro_batch, args.accumulate, args.time_budget, args.eval_every) < 1:
        parser.error("Training counts and time budget must be positive")
    args.output.mkdir(parents=True, exist_ok=True)
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    torch.set_num_threads(2)
    assert torch.cuda.is_available(), "Enable GPU on Kaggle"
    manifest = json.loads((args.output / "manifest.json").read_text())
    train, val = manifest["train"], manifest["val"]

    from inpainter_setting.torch_utils.ops import bias_act, upfirdn2d
    bias_act._init = lambda: False
    upfirdn2d._init = lambda: False
    logging.getLogger("mmcv").setLevel(logging.WARNING)
    config = Config.fromfile("config/fosp_config.py")
    config.model.pretrained = None
    model = build_segmentor(config.model).cuda()
    model.init_weights()
    from torch_utils.ops import bias_act as ba, upfirdn2d as uf
    ba._init = lambda: False
    uf._init = lambda: False
    checkpoint_path = args.output / "last.pth"
    if not (args.resume and checkpoint_path.is_file()) and not args.evaluate_test:
        mat = torch.load(args.weights / "Places_512_FullData_G.pth", map_location="cpu", weights_only=True)
        mat = mat.get("state_dict", mat)
        required = model.decode_head.G.state_dict()
        assert set(required).issubset(mat), "Missing retained MAT tensors"
        assert all(mat[k].shape == v.shape for k, v in required.items())
        model.decode_head.G.load_state_dict({k: mat[k] for k in required}, strict=True)
        del mat, required
        backbone = torch.load(args.weights / "mit_b3.pth", map_location="cpu", weights_only=True)
        backbone = backbone.get("state_dict", backbone)
        backbone = {k.removeprefix("backbone."): v for k, v in backbone.items() if not k.startswith("decode_head.")}
        model.backbone.load_state_dict(backbone, strict=True)
        del backbone

    # Match the upstream learning-rate/weight-decay parameter rules.
    groups = []
    parameters = []
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            multiplier = 10.0 if "head" in name else 1.0
            no_decay = "norm" in name or "pos_block" in name
            groups.append({"params": [parameter], "lr": 6e-5 * multiplier,
                           "lr_multiplier": multiplier, "weight_decay": 0.0 if no_decay else 0.01})
            parameters.append(parameter)
    optimizer = torch.optim.AdamW(groups, lr=6e-5, weight_decay=0.01)
    start, best, history = 0, -1.0, []
    if args.evaluate_test:
        checkpoint = torch.load(args.output / "best.pth", map_location="cpu", weights_only=True)
        model.load_state_dict(checkpoint["state_dict"], strict=True)
        scores = evaluate(model, manifest["test"], 0.4, args.output, "test", preview=True)
        (args.output / "test_metrics.json").write_text(json.dumps(scores, indent=2))
        print("SMOKE5K_TEST", json.dumps(scores), flush=True)
        return
    if args.resume and checkpoint_path.is_file():
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        assert checkpoint["run_config"]["manifest_sha256"] == hashlib.sha256((args.output / "manifest.json").read_bytes()).hexdigest()
        for key, value in (("max_updates", args.max_updates), ("micro_batch", args.micro_batch), ("accumulate", args.accumulate)):
            assert checkpoint["run_config"][key] == value, f"Resume config changed: {key}"
        model.load_state_dict(checkpoint["state_dict"], strict=True)
        optimizer.load_state_dict(checkpoint["optimizer"])
        start, best, history = checkpoint["update"], checkpoint["best_f1"], checkpoint["history"]
        random.setstate(checkpoint["rng"]["python"])
        state = checkpoint["rng"]["numpy"]
        np.random.set_state((state[0], np.array(state[1], dtype=np.uint32), *state[2:]))
        torch.set_rng_state(checkpoint["rng"]["torch"])
        torch.cuda.set_rng_state_all(checkpoint["rng"]["cuda"])
        del checkpoint

    run_config = {"dataset": "SMOKE5K", "architecture": "official FoSp MiT-B3",
                  "source_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
                  "manifest_sha256": hashlib.sha256((args.output / "manifest.json").read_bytes()).hexdigest(),
                  "max_updates": args.max_updates, "micro_batch": args.micro_batch,
                  "accumulate": args.accumulate, "effective_batch": args.micro_batch * args.accumulate,
                  "size": 512, "seed": 42, "threshold": 0.4, "test_used_for_selection": False,
                  "reference_mat_ops": True, "paper_reproduction": False,
                  "differences": ["train-only validation holdout", "train/test exact overlap removed",
                                  "microbatch BN statistics differ from physical batch 6", "simplified mild zoom/photometric augmentation"],
                  "metric_note": "F1 beta=1, fixed threshold 0.4; mean per image and global pixels; native mask size. Not assumed equal to paper F_beta."}
    (args.output / "run_config.json").write_text(json.dumps(run_config, indent=2))
    shutil.copy2(__file__, args.output / "train_smoke5k.py")
    if start >= args.max_updates:
        (args.output / "status.json").write_text(json.dumps({"state": "complete", "update": start,
            "target_updates": args.max_updates, "best_validation_f1": best, "test_evaluated": False}, indent=2))
        print("SMOKE5K_SCHEDULE_ALREADY_COMPLETE", start, flush=True)
        return
    began = time.monotonic()
    meta = {"img_shape": (512, 512, 3), "ori_shape": (512, 512, 3), "pad_shape": (512, 512, 3),
            "scale_factor": 1.0, "flip": False, "img_norm_cfg": {"mean": MEAN, "std": STD, "to_rgb": True}}

    def save(update, scores=None):
        state = np.random.get_state()
        payload = {"state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                   "optimizer": optimizer.state_dict(), "update": update, "best_f1": best,
                   "run_config": run_config, "history": history,
                   "rng": {"python": random.getstate(), "numpy": (state[0], state[1].tolist(), *state[2:]),
                           "torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all()}}
        atomic_save(payload, checkpoint_path)
        if scores is not None and scores["image_mean"]["f1"] >= best:
            atomic_save({"state_dict": payload["state_dict"], "update": update,
                         "run_config": run_config, "validation": scores}, args.output / "best.pth")
        print("CHECKPOINT_SAVED", update, str(checkpoint_path), flush=True)

    print("SMOKE5K_TRAIN_START", json.dumps(run_config | {"train_images": len(train), "val_images": len(val), "start_update": start}), flush=True)
    for update in range(start + 1, args.max_updates + 1):
        model.train()
        model.decode_head.G.eval()
        factor = (1 - (update - 1) / args.max_updates) * ((1e-6 + (1 - 1e-6) * update / 1500) if update < 1500 else 1.0)
        for group in optimizer.param_groups:
            group["lr"] = 6e-5 * group["lr_multiplier"] * factor
        optimizer.zero_grad(set_to_none=True)
        loss_sum = 0.0
        for _ in range(args.accumulate):
            rows = random.sample(train, args.micro_batch)
            pairs = [load_pair(row, augment=True) for row in rows]
            x = torch.stack([p[0] for p in pairs]).cuda()
            y = torch.stack([p[1] for p in pairs]).cuda()
            losses = model(img=x, img_metas=[meta] * len(rows), gt_semantic_seg=y, return_loss=True)
            loss, _ = model._parse_losses(losses)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss at update {update}")
            (loss / args.accumulate).backward()
            loss_sum += float(loss.detach()) / args.accumulate
        torch.nn.utils.clip_grad_norm_(parameters, 1.0, error_if_nonfinite=True)
        optimizer.step()
        seconds = time.monotonic() - began
        if update == start + 1 or update % 25 == 0:
            print("UPDATE", update, "LOSS", round(loss_sum, 6), "SECONDS", round(seconds, 1),
                  "PEAK_GPU_GB", round(torch.cuda.max_memory_allocated() / 1024**3, 2), flush=True)
            (args.output / "status.json").write_text(json.dumps({"state": "running", "update": update,
                "target_updates": args.max_updates, "elapsed_seconds": seconds, "loss": loss_sum,
                "seconds_per_update": seconds / (update - start)}, indent=2))
        final = update == args.max_updates or seconds >= args.time_budget
        if update % args.eval_every == 0 or final:
            scores = evaluate(model, val, 0.4, args.output, "validation", preview=True)
            history.append({"update": update, "loss": loss_sum, **scores})
            (args.output / "validation_history.json").write_text(json.dumps(history, indent=2))
            best = max(best, scores["image_mean"]["f1"])
            print("SMOKE5K_VALIDATION", json.dumps(history[-1]), flush=True)
            save(update, scores)
        elif update == 25 or update % 100 == 0:
            save(update)
        if final:
            status = {"state": "complete" if update == args.max_updates else "time_budget_checkpointed",
                      "update": update, "target_updates": args.max_updates, "best_validation_f1": best,
                      "test_evaluated": False}
            (args.output / "status.json").write_text(json.dumps(status, indent=2))
            print("SMOKE5K_SEGMENT_DONE", json.dumps(status), flush=True)
            break


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/smoke5k-code/bootstrap_kaggle.py
"""Prepare an isolated FoSp environment, download official SMOKE5K, and train.

Intended for a GPU Kaggle Python notebook with Internet enabled. Does not replace
Kaggle's PyTorch installation. Place all three .py files together.
"""
import argparse
import hashlib
import os
import shutil
import stat
import subprocess
import sys
import zipfile
from pathlib import Path


BASE = Path("/kaggle/working")
REPO = BASE / "FoSp-official"
COMMIT = "02b9fd18ad1930d1081cf3b35a569d4b59e71d63"
DEPS = BASE / "fosp-deps"
MMCV = BASE / "mmcv-1.6.2-source"
WEIGHTS = BASE / "fosp-pretrained"
OUTPUT = BASE / "fosp_smoke5k"


def run(command, **kwargs):
    return subprocess.run(command, check=True, **kwargs)


def download(url, path):
    import requests
    if path.is_file():
        return
    temporary = path.with_suffix(path.suffix + ".part")
    with requests.get(url, stream=True, timeout=(30, 180)) as response:
        response.raise_for_status()
        with temporary.open("wb") as stream:
            for chunk in response.iter_content(1024 * 1024):
                stream.write(chunk)
    os.replace(temporary, path)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--prepare-only", action="store_true")
    parser.add_argument("--resume-from", type=Path)
    parser.add_argument("--max-updates", type=int, default=40000)
    parser.add_argument("--time-budget", type=int, default=28800)
    args = parser.parse_args()
    import torch
    assert torch.cuda.is_available(), "Enable a GPU accelerator first"
    code = Path(__file__).resolve().parent
    BASE.mkdir(exist_ok=True)
    DEPS.mkdir(exist_ok=True)
    WEIGHTS.mkdir(exist_ok=True)
    OUTPUT.mkdir(exist_ok=True)
    if not (REPO / ".git").is_dir():
        run(["git", "init", str(REPO)])
        run(["git", "remote", "add", "origin", "https://github.com/LujianYao/FoSp.git"], cwd=REPO)
        run(["git", "fetch", "--depth", "1", "origin", COMMIT], cwd=REPO)
        run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)
    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
    assert actual == COMMIT, f"Unexpected FoSp source revision: {actual}"
    if not MMCV.is_dir():
        run(["git", "clone", "--depth", "1", "--branch", "v1.6.2", "https://github.com/open-mmlab/mmcv.git", str(MMCV)])
    run([sys.executable, "-m", "pip", "install", "--target", str(DEPS), "--no-deps",
         "setuptools==80.9.0", "wheel", "addict", "yapf==0.40.1", "easydict", "future", "ninja", "timm", "gdown"])
    env = os.environ.copy()
    paths = [str(REPO), str(MMCV), str(DEPS)]
    paths += [p for p in sys.path if p and ("site-packages" in p or "dist-packages" in p)]
    env["PYTHONPATH"] = os.pathsep.join(paths)
    run([sys.executable, "-c", "import setuptools,torch,mmcv; from mmseg.models import build_segmentor; print('FOSP_IMPORT_OK',torch.__version__,mmcv.__version__,flush=True)"], cwd=REPO, env=env)
    archive = BASE / "SMOKE5K-official.zip"
    if not archive.is_file():
        run([sys.executable, "-c", "import gdown,sys; gdown.download(id='11TM8hsh9R6ZTvLAUzfD6eD051MbOufCi',output=sys.argv[1],quiet=False)", str(archive)], cwd=REPO, env=env)
    assert zipfile.is_zipfile(archive), "Invalid SMOKE5K ZIP download"
    root = BASE / "SMOKE5K-official"
    root.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as bundle:
        for item in bundle.infolist():
            assert (root / item.filename).resolve().is_relative_to(root.resolve())
            assert not stat.S_ISLNK(item.external_attr >> 16)
        bundle.extractall(root)
    download("https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b3_20220624-13b1141c.pth", WEIGHTS / "mit_b3.pth")
    mat_path = WEIGHTS / "Places_512_FullData_G.pth"
    download("https://github.com/Sanster/models/releases/download/add_mat/Places_512_FullData_G.pth", mat_path)
    assert hashlib.sha256(mat_path.read_bytes()).hexdigest() == "0512e37ebba3986b0355130b2e2c1f95736d0778ac82e91b1212b4b21c231312", "Unexpected MAT weights"
    for filename in ("prepare_smoke5k.py", "train_smoke5k.py"):
        if (code / filename).resolve() != (REPO / filename).resolve():
            shutil.copy2(code / filename, REPO / filename)
    run([sys.executable, "-u", str(REPO / "prepare_smoke5k.py")], cwd=REPO, env=env)
    if args.resume_from:
        assert (args.resume_from / "last.pth").is_file(), "Resume folder must contain last.pth"
        assert (args.resume_from / "manifest.json").read_bytes() == (OUTPUT / "manifest.json").read_bytes(), "Resume dataset manifest differs"
        for filename in ("last.pth", "best.pth", "validation_history.json"):
            source, target = args.resume_from / filename, OUTPUT / filename
            if source.is_file() and source.resolve() != target.resolve():
                assert not target.exists(), f"Refusing to overwrite existing {target}"
                shutil.copy2(source, target)
    if args.prepare_only:
        return
    run([sys.executable, "-u", str(REPO / "train_smoke5k.py"), "--resume",
         "--max-updates", str(args.max_updates), "--time-budget", str(args.time_budget),
         "--micro-batch", "2", "--accumulate", "3", "--eval-every", "250"], cwd=REPO, env=env)
    import json
    status = json.loads((OUTPUT / "status.json").read_text())
    if status["state"] == "complete":
        run([sys.executable, "-u", str(REPO / "train_smoke5k.py"), "--evaluate-test"], cwd=REPO, env=env)
    print("SMOKE5K_PACKAGE", shutil.make_archive(str(OUTPUT), "zip", root_dir=OUTPUT), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
import subprocess, sys
candidates = list(Path("/kaggle/input").rglob("fosp_smoke5k_14000/last.pth"))
assert len(candidates) == 1, "Attach Output Version 4 containing fosp_smoke5k_14000"
RESUME_FROM = candidates[0].parent
command = [sys.executable,"-u","/kaggle/working/smoke5k-code/bootstrap_kaggle.py","--max-updates","40000","--time-budget","28800"]
if RESUME_FROM:
    command += ["--resume-from",str(RESUME_FROM)]
subprocess.run(command,check=True)

In [ ]:
import json
from IPython.display import display,Image
output=Path("/kaggle/working/fosp_smoke5k")
print((output/"status.json").read_text())
history=json.loads((output/"validation_history.json").read_text())
print(json.dumps(history[-1],indent=2))
if (output/"test_metrics.json").exists():
    print("FINAL TEST",(output/"test_metrics.json").read_text())
display(Image(filename=str(output/"validation_preview.jpg")))
print("Save a Kaggle version WITH OUTPUTS before ending the session.")